# 词嵌入与位置编码
## Embeddings and Positional Encoding

<img src="../images/logo.png" width=150>

词嵌入（Word Embedding）将离散的自然语言符号映射到连续的向量空间，是LLM理解语义的基石。位置编码（Positional Encoding）则为序列中的每个位置提供独特表示，使模型能够区分词的顺序关系。

Word embeddings map discrete natural language symbols to continuous vector spaces, forming the foundation of LLM semantic understanding. Positional encoding provides unique representations for each position in the sequence, enabling the model to distinguish word order relationships.

# 1. Token Embedding
## 1. Token Embedding

In [ ]:
import torch
import torch.nn as nn
import math
import matplotlib.pyplot as plt
import numpy as np

class TokenEmbedding(nn.Module):
    """
    标准词嵌入层
    Standard token embedding layer
    """
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.embed_dim = embed_dim
        self._init_weights()
    
    def _init_weights(self):
        # 使用截断正态分布初始化 / Initialize with truncated normal distribution
        nn.init.trunc_normal_(self.embedding.weight, mean=0.0, std=0.02)
    
    def forward(self, tokens):
        """tokens: (batch, seq_len) -> (batch, seq_len, embed_dim)"""
        return self.embedding(tokens)

# 测试 / Test
vocab_size = 50000
embed_dim = 768
token_emb = TokenEmbedding(vocab_size, embed_dim)

batch_size = 2
seq_len = 10
tokens = torch.randint(0, vocab_size, (batch_size, seq_len))
embeddings = token_emb(tokens)

print(f"Input tokens shape: {tokens.shape}")
print(f"Output embeddings shape: {embeddings.shape}")
print(f"Embedding dimension: {embed_dim}")

# 2. 位置编码类型对比
## 2. Positional Encoding Types Comparison

In [ ]:
class AbsolutePositionalEncoding(nn.Module):
    """
    绝对位置编码（GPT-2使用）
    Absolute positional encoding (used by GPT-2)
    """
    def __init__(self, max_seq_len, embed_dim):
        super().__init__()
        # 可学习的绝对位置编码 / Learnable absolute positional encoding
        self.position_embedding = nn.Embedding(max_seq_len, embed_dim)
    
    def forward(self, seq_len):
        """返回位置嵌入 / Return position embeddings"""
        position_ids = torch.arange(seq_len).unsqueeze(0)  # (1, seq_len)
        return self.position_embedding(position_ids)


class SinusoidalPositionalEncoding(nn.Module):
    """
    正弦位置编码（原始Transformer使用）
    Sinusoidal positional encoding (used by original Transformer)
    """
    def __init__(self, embed_dim, max_seq_len=2048):
        super().__init__()
        self.embed_dim = embed_dim
        
        # 预计算位置编码 / Precompute positional encodings
        position = torch.arange(max_seq_len).unsqueeze(1)  # (max_seq_len, 1)
        div_term = torch.exp(torch.arange(0, embed_dim, 2) * (-math.log(10000.0) / embed_dim))
        
        pe = torch.zeros(max_seq_len, embed_dim)
        pe[:, 0::2] = torch.sin(position * div_term)  # 偶数维度 / Even dimensions
        pe[:, 1::2] = torch.cos(position * div_term)  # 奇数维度 / Odd dimensions
        
        self.register_buffer('pe', pe)
    
    def forward(self, seq_len):
        """返回位置编码 / Return positional encoding"""
        return self.pe[:seq_len].unsqueeze(0)  # (1, seq_len, embed_dim)


# 测试三种位置编码 / Test three types of positional encodings
max_seq_len = 128
embed_dim = 64

abs_pe = AbsolutePositionalEncoding(max_seq_len, embed_dim)
sin_pe = SinusoidalPositionalEncoding(embed_dim, max_seq_len)

# 可视化 / Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 绝对位置编码可视化 / Absolute positional encoding visualization
abs_emb = abs_pe(max_seq_len).squeeze()  # (seq_len, embed_dim)
im1 = axes[0].imshow(abs_emb[:50].T, aspect='auto', cmap='viridis')
axes[0].set_title('Learned Absolute Positional Encoding')
axes[0].set_xlabel('Position')
axes[0].set_ylabel('Dimension')
plt.colorbar(im1, ax=axes[0])

# 正弦位置编码可视化 / Sinusoidal positional encoding visualization
sin_emb = sin_pe(max_seq_len).squeeze()
im2 = axes[1].imshow(sin_emb[:50].T, aspect='auto', cmap='viridis')
axes[1].set_title('Sinusoidal Positional Encoding')
axes[1].set_xlabel('Position')
axes[1].set_ylabel('Dimension')
plt.colorbar(im2, ax=axes[1])

plt.tight_layout()
plt.savefig('../images/positional_encoding.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to ../images/positional_encoding.png")

# 3. RoPE（旋转位置编码）
## 3. RoPE (Rotary Position Embedding)

In [ ]:
class RotaryPositionalEmbedding(nn.Module):
    """
    旋转位置编码（RoPE）- LLaMA、GLM等模型使用
    Rotary Positional Embedding - used by LLaMA, GLM, etc.
    """
    def __init__(self, dim, max_seq_len=2048, base=10000):
        super().__init__()
        self.dim = dim
        self.max_seq_len = max_seq_len
        self.base = base
        
        # 预计算旋转角度 / Precompute rotation angles
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        t = torch.arange(max_seq_len)
        freqs = torch.outer(t, inv_freq)  # (seq_len, dim/2)
        
        # 缓存cos和sin / Cache cos and sin
        self.register_buffer('cos_cached', freqs.cos())
        self.register_buffer('sin_cached', freqs.sin())
    
    def forward(self, seq_len):
        """返回旋转矩阵 / Return rotation matrices"""
        return (
            self.cos_cached[:seq_len],
            self.sin_cached[:seq_len]
        )

def apply_rotary_pos_emb(q, k, cos, sin):
    """
    应用RoPE到Q和K
    Apply RoPE to query and key
    
    q: (batch, heads, seq, head_dim)
    k: (batch, heads, seq, head_dim)
    cos, sin: (seq, head_dim/2)
    
    修复：标准RoPE对最后两维进行两两旋转，而不是简单切分
    """
    # 确保cos和sin是正确形状 / Ensure cos and sin have correct shape
    if cos.dim() == 2:
        cos = cos.unsqueeze(0).unsqueeze(0)  # (1, 1, seq, head_dim/2)
        sin = sin.unsqueeze(0).unsqueeze(0)
    
    # 旋转是对每对维度进行的：dim 0&1, 2&3, 4&5, ...
    # Rotation is applied to pairs: dim 0&1, 2&3, 4&5, ...
    half_dim = cos.shape[-1]
    
    # 分离奇偶位置 / Separate odd and even positions
    q0 = q[..., :half_dim]  # dim 0, 2, 4, ...
    q1 = q[..., half_dim:]  # dim 1, 3, 5, ...
    k0 = k[..., :half_dim]
    k1 = k[..., half_dim:]
    
    # RoPE公式: 偶数维 * cos - 奇数维 * sin, 偶数维 * sin + 奇数维 * cos
    # RoPE formula
    q_embed = torch.cat([q0 * cos - q1 * sin, q0 * sin + q1 * cos], dim=-1)
    k_embed = torch.cat([k0 * cos - k1 * sin, k0 * sin + k1 * cos], dim=-1)
    
    return q_embed, k_embed

In [ ]:
# 词嵌入相似度可视化 / Word Embedding Similarity Visualization
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Word similarity heatmap / 词相似度热力图
ax1 = axes[0]
words = ['king', 'queen', 'man', 'woman', 'prince', 'princess', 'boy', 'girl']
# Simulated embedding similarities
sim_matrix = np.array([
    [1.00, 0.85, 0.70, 0.65, 0.80, 0.75, 0.50, 0.45],
    [0.85, 1.00, 0.60, 0.75, 0.78, 0.82, 0.48, 0.52],
    [0.70, 0.60, 1.00, 0.55, 0.65, 0.50, 0.80, 0.40],
    [0.65, 0.75, 0.55, 1.00, 0.50, 0.65, 0.35, 0.75],
    [0.80, 0.78, 0.65, 0.50, 1.00, 0.70, 0.60, 0.40],
    [0.75, 0.82, 0.50, 0.65, 0.70, 1.00, 0.35, 0.55],
    [0.50, 0.48, 0.80, 0.35, 0.60, 0.35, 1.00, 0.30],
    [0.45, 0.52, 0.40, 0.75, 0.40, 0.55, 0.30, 1.00]
])

im = ax1.imshow(sim_matrix, cmap='RdYlGn', vmin=0, vmax=1)
ax1.set_xticks(range(len(words)))
ax1.set_yticks(range(len(words)))
ax1.set_xticklabels(words, rotation=45)
ax1.set_yticklabels(words)
ax1.set_title('Word Embedding Similarity
(Cosine Similarity)')
plt.colorbar(im, ax=ax1)

# 2. king - man + woman ≈ queen visualization
ax2 = axes[1]
vectors = {
    'king': np.array([0.8, 0.2]),
    'man': np.array([0.3, 0.7]),
    'woman': np.array([0.2, 0.9]),
    'queen': np.array([0.7, 0.85]),
    'result': np.array([0.7, 0.45])  # king - man + woman
}

for word, vec in vectors.items():
    color = 'green' if word == 'result' else 'blue'
    marker = '*' if word == 'result' else 'o'
    ax2.scatter(vec[0], vec[1], s=200, marker=marker, label=word, color=color, zorder=5)
    ax2.annotate(word, (vec[0]+0.03, vec[1]+0.03), fontsize=12)

ax2.annotate('', xy=vectors['king'], xytext=vectors['man'],
            arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))
ax2.annotate('', xy=vectors['woman'], xytext=vectors['man'],
            arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))
ax2.set_xlim(0, 1)
ax2.set_ylim(0, 1)
ax2.set_xlabel('Dimension 1')
ax2.set_ylabel('Dimension 2')
ax2.set_title('Word2Vec Analogy: king - man + woman ≈ queen')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../images/word_embedding_similarity.png', dpi=150, bbox_inches='tight')
plt.show()

print("Word embedding visualization saved!")

# 4. ALiBi（线性偏置注意力）
## 4. ALiBi (Attention with Linear Biases)

In [ ]:
def alibi_slopes(num_heads):
    """
    生成ALiBi的斜率系数
    Generate ALiBi slope coefficients
    
    ALiBi为每个头分配不同的斜率，使注意力能够区分距离
    ALiBi assigns different slopes to each head, allowing attention to distinguish distances
    
    修复：
    1. math.log2(n, 2) -> math.log2(n) (第一个参数是要计算对数的数，第二个才是底数)
    2. 公式从 start * (i ** ...) 修正为 2 ** (-8 * (i+1) / n) 的几何序列
    """
    # 论文中的公式: slope = 2^(-8/n)^(i+1) 或等价的 2^(-8*(i+1)/n)
    # 简化写法：2^(-8/n), 2^(-16/n), ..., 2^(-8*num_heads/n)
    n = num_heads
    # 基础斜率 = 2^(-8/n)
    base = 2 ** (-8 / n)
    
    # 各头的斜率构成几何序列: base^1, base^2, ..., base^n
    return [2 ** (-8 * (i + 1) / n) for i in range(num_heads)]

def build_alibi_mask(seq_len, num_heads):
    """
    构建ALiBi注意力掩码（向量化实现）
    Build ALiBi attention mask (vectorized implementation)
    """
    slopes = torch.tensor(alibi_slopes(num_heads))  # (num_heads,)

    # 创建相对距离矩阵 / Create relative distance matrix
    positions = torch.arange(seq_len)  # (seq_len,)
    diff = positions.unsqueeze(1) - positions.unsqueeze(0)  # (seq_len, seq_len)
    diff = diff.abs()  # (seq_len, seq_len)

    # 扩展斜率到多头 / Expand slopes to multi-head
    slopes = slopes.view(num_heads, 1, 1)  # (num_heads, 1, 1)
    diff = diff.unsqueeze(0)  # (1, seq_len, seq_len)

    # 计算ALiBi偏置 / Compute ALiBi bias
    alibi_bias = -slopes * diff  # (num_heads, seq_len, seq_len)

    # 上三角设置为-inf / Set upper triangle to -inf
    mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
    alibi_bias[:, mask] = float('-inf')

    return alibi_bias  # (num_heads, seq_len, seq_len)

# 5. 完整的Embedding模块
## 5. Complete Embedding Module

In [ ]:
class CombinedEmbedding(nn.Module):
    """
    组合词嵌入和位置编码
    Combined token and positional embedding
    支持多种位置编码方式
    Supports multiple positional encoding methods
    """
    def __init__(self, vocab_size, embed_dim, max_seq_len, pos_encoding_type='rope'):
        super().__init__()
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.pos_encoding_type = pos_encoding_type

        # Token嵌入 / Token embedding
        self.token_embedding = TokenEmbedding(vocab_size, embed_dim)

        # 位置编码（根据类型选择）/ Positional encoding (select by type)
        if pos_encoding_type == 'learned':
            self.position_embedding = AbsolutePositionalEncoding(max_seq_len, embed_dim)
        elif pos_encoding_type == 'sinusoidal':
            self.position_embedding = SinusoidalPositionalEncoding(embed_dim, max_seq_len)
        elif pos_encoding_type == 'rope':
            self.position_embedding = RotaryPositionalEmbedding(embed_dim, max_seq_len)
        else:
            raise ValueError(f"Unknown positional encoding type: {pos_encoding_type}")

    def forward(self, tokens):
        """
        tokens: (batch, seq_len)
        returns: (batch, seq_len, embed_dim)
        """
        # Token嵌入 / Token embedding
        x = self.token_embedding(tokens)

        # 位置编码 / Positional encoding
        if self.pos_encoding_type in ['learned', 'sinusoidal']:
            pos_emb = self.position_embedding(tokens.size(1))
            x = x + pos_emb
        # RoPE不会在这里添加，会在attention层应用
        # RoPE is applied in attention layer, not added here

        return x

    def get_rope_cos_sin(self, seq_len):
        """获取RoPE的cos和sin，用于在attention层应用RoPE
        修复：RoPE类型需要生成缓存，供外部attention模块调用
        """
        if self.pos_encoding_type == 'rope':
            return self.position_embedding(seq_len)
        return None

# 6. 嵌入空间可视化
## 6. Embedding Space Visualization

In [ ]:
# 可视化不同词的嵌入分布
# Visualize embedding distribution of different words

vocab_size = 1000
embed_dim = 32
emb = TokenEmbedding(vocab_size, embed_dim)

# 获取部分词的嵌入 / Get embeddings for subset of words
word_ids = torch.arange(100)
embeddings = emb(word_ids)

# 计算嵌入的统计信息 / Compute embedding statistics
embed_norms = embeddings.norm(dim=-1).numpy()
embed_means = embeddings.mean(dim=0).numpy()

# 可视化 / Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(embed_norms, bins=20, edgecolor='black')
axes[0].set_title('Embedding Norms Distribution')
axes[0].set_xlabel('L2 Norm')
axes[0].set_ylabel('Frequency')

axes[1].plot(embed_means)
axes[1].set_title('Mean Embedding per Dimension')
axes[1].set_xlabel('Dimension')
axes[1].set_ylabel('Mean Value')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../images/embedding_stats.png', dpi=150, bbox_inches='tight')
plt.show()

# 总结 Summary

| 类型 | 使用模型 | 特点 |
|------|----------|------|
| Learned | GPT-2 | 可学习，简单但有长度限制 |
| Sinusoidal | 原始Transformer | 不用学习，可外推但效果一般 |
| RoPE | LLaMA, GLM, DeepSeek | 无需添加，直接应用在Q/K |
| ALiBi | 多个模型 | 线性偏置，训练和推理一致 |

| Type | Used by | Characteristics |
|------|---------|----------------|
| Learned | GPT-2 | Learnable, simple but length-limited |
| Sinusoidal | Original Transformer | No learning, extrapolatable but average效果 |
| RoPE | LLaMA, GLM, DeepSeek | No addition, applied directly to Q/K |
| ALiBi | Various | Linear bias, consistent training and inference |

# 已实现 / Implemented

本notebook已完整实现以下内容：

1. **Token Embedding** - 可学习的词嵌入层
2. **绝对位置编码** - GPT-2风格的可学习位置编码
3. **Sinusoidal位置编码** - 原始Transformer使用的正弦编码
4. **RoPE旋转位置编码** - LLaMA/GLM使用的旋转编码
5. **ALiBi线性偏置注意力** - 无需位置编码的注意力方式
6. **组合Embedding模块** - 支持多种位置编码方式

## 扩展阅读 / Further Reading

如需深入学习，可进一步探索：

| 主题 | 说明 | 推荐资源 |
|------|------|----------|
| **Flash Attention** | 加速注意力计算，减少显存 | [Flash Attention Paper](https://arxiv.org/abs/2205.14135) |
| **Paged Attention** | vLLM使用的显存管理技术 | [vLLM Paper](https://arxiv.org/abs/2309.06180) |
| **YaRN** | RoPE的长上下文外推增强 | [YaRN Paper](https://arxiv.org/abs/2309.00071) |
| **NTK-aware Scaling** | 改进的长上下文插值 | [NTK Scaling](https://www.reddit.com/r/LocalLLaMA/comments/14nm3pd/ntkaware_scaling_rovers_dont_need_to_cut_off/) |
